# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment below in a notebook environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display core metadata
md = dataset.metadata
print(f"Dataset: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Version: {md.version}")
print(f"Identifier: {md.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Find all record sets and fields, referencing by their `@id`s

print('Available record sets and their fields:')
all_record_sets = list(dataset.record_sets)
for rs in all_record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    for field in rs.fields:
        info = f"    - Field: {getattr(field, 'name', getattr(field, 'id', 'N/A'))}, @id: {getattr(field, 'id', 'N/A')}, type: {getattr(field, 'data_type', '-')}")
        print(info)
if not all_record_sets:
    print("No recordSet information present in top-level metadata. Attempting to load records using inferred @id ...")
else:
    print(f"Total record sets: {len(all_record_sets)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
from pprint import pprint

# If there are explicit record sets, extract their IDs
record_set_ids = []
if all_record_sets:
    record_set_ids = [rs.id for rs in all_record_sets]
else:
    # Fallback: try commonly used record set IDs or from Croissant spec
    record_set_ids = ['cr:RecordSet', 'dataset']  # May need to adjust for new datasets

# Attempt to load data for each record set ID
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
        else:
            print(f"No records found for record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Could not load records for '@id': {record_set_id}: {e}")

if not dataframes:
    print('No tabular record sets found or the dataset contains only metadata.')
else:
    # Preview the first loaded DataFrame (use the first available record set)
    first_key = list(dataframes.keys())[0]
    print(f"Available columns for record set '@id'={first_key}:")
    pprint(dataframes[first_key].columns.tolist())
    dataframes[first_key].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This may include removing outliers, transforming data distributions, or grouping data for downstream analysis.

In [ ]:
# For demonstration, pick a numeric field if one exists
df = None
record_set_id = None
for key, frame in dataframes.items():
    if not frame.empty:
        # Find a plausible numeric column
        num_col = None
        for col in frame.columns:
            if pd.api.types.is_numeric_dtype(frame[col]):
                num_col = col
                break
        if num_col:
            df = frame
            record_set_id = key
            numeric_field = num_col
            break

if df is not None:
    print(f"Using record set '@id': {record_set_id}, numeric field: '{numeric_field}'")
    # Filter records where numeric_field > threshold
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized values for column '{numeric_field}':")
    print(filtered_df[[numeric_field, norm_col]].head())

    # If any other non-numeric field exists, group by it
    cat_col = None
    for col in filtered_df.columns:
        if col != numeric_field and pd.api.types.is_object_dtype(filtered_df[col]):
            cat_col = col
            break
    if cat_col is not None:
        grouped = filtered_df.groupby(cat_col)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {cat_col}:")
        print(grouped.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize numeric data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '@id': {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

In this notebook, we:
- Loaded metadata and any available record sets from the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`.
- Explored available record sets and their fields, referencing all entities by their `@id` per best practices.
- Loaded tabular data into pandas for further analysis and demonstrated basic exploratory data analysis and a simple visualization.

For more advanced use cases—such as joining across record sets, advanced processing, or leveraging field annotations—cross-reference all field and record set lookups with their `@id` values as demonstrated above.

Learn more:
- [mlcroissant documentation](https://mlcommons.github.io/croissant/)  
- [Original dataset record](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
